# Modul B · Kapitel 1.7 — ReAct

## Reasoning und Werkzeuge zu einem Agent-Loop verbinden

**Lernziel:** Du baust eine kleine ReAct-Schleife, in der das Modell den nächsten Werkzeugaufruf auswählt, Python ihn kontrolliert ausführt und das Ergebnis als Observation zurückgibt.

Durchgehende Frage: *Are we affected by CVE-2026-3224, and what should we do?*

```
Thought → Action → Werkzeug → Observation ─┐
   ↑                                      │
   └──────────────────────────────────────┘
                         ↓
                    Final Answer
```

Das Notebook enthält **drei zentrale Challenges**. Führe die Zellen von oben nach unten aus.


---
## 0 · Setup

Das Notebook verwendet das gemeinsame Modell aus `helfer.py` und zwei lokale Beispieldatensätze. Standardmäßig läuft `qwen3.5:0.8b` über Ollama.


In [ ]:
# ▶️ Pakete, Pfade und Helfer laden
import importlib
import re
import sys
import textwrap
from pathlib import Path

try:
    import openai
except ImportError:
    %pip install -q openai
    import openai

for kandidat in [Path.cwd(), *Path.cwd().parents, Path("/content"),
                 Path("/content/01_prompt-engineering")]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

import helfer
importlib.reload(helfer)
from helfer import BASIS_URL, MODELL, REASONING, client, frage_llm, lade_daten, zeige

print(f"Server: {BASIS_URL}")
print(f"Modell: {MODELL}")


In [ ]:
# ▶️ Modelzugang und Daten prüfen
print("Testantwort:", frage_llm("Reply with the single word: ready"))

CVES = lade_daten("cve_datenbank")
HOSTS = lade_daten("host_inventory")
FRAGE = "Are we affected by CVE-2026-3224, and what should we do?"

print(f"CVEs: {len(CVES)} | Systeme: {len(HOSTS)}")
print("Frage:", FRAGE)


---
## 1 · Warum ReAct?

Das Modell kennt weder unsere erfundene CVE-Datenbank noch unser internes Inventar. Ein längerer Prompt kann fehlende Daten nicht ersetzen. ReAct gibt dem Modell deshalb Werkzeuge und trennt klar die Verantwortlichkeiten:

| Bestandteil | Verantwortlich | Zweck |
|---|---|---|
| `Thought` | Modell | nächsten sinnvollen Schritt wählen |
| `Action` | Modell | Werkzeug und Argument anfordern |
| `Observation` | **Programm** | echtes Werkzeugergebnis zurückgeben |
| `Final Answer` | Modell | belegte Antwort formulieren |

Der entscheidende Punkt: **Das Modell darf eine Observation niemals selbst erfinden.**


In [ ]:
# ▶️ Ohne Werkzeuge fehlt die interne Faktenbasis
antwort_ohne = frage_llm(FRAGE)
zeige(antwort_ohne, titel="Antwort ohne Werkzeuge")


---
## 2 · Die Werkzeuge

Ein Werkzeug ist hier nur eine Python-Funktion: ein Argument hinein, lesbarer Text zurück. Fehler kommen ebenfalls als Text zurück. So kann die Schleife reagieren, statt durch eine Exception abzubrechen.


In [ ]:
# ▶️ Zwei Werkzeuge über den lokalen Beispieldaten
def cve_lookup(cve_id):
    """Liefert die entscheidenden Fakten zu einer CVE."""
    eintrag = CVES.get(cve_id.strip().upper())
    if eintrag is None:
        return f"ERROR: no entry for '{cve_id}'. Known ids: {', '.join(sorted(CVES))}"
    return (
        f"{eintrag['id']} | severity {eintrag['severity']} (CVSS {eintrag['cvss']}) "
        f"| component {eintrag['component']} | affected {eintrag['affected_versions']} "
        f"| fixed in {eintrag['fixed_version']} | action {eintrag['recommended_action']}"
    )


def host_inventory(component):
    """Liefert unsere Systeme für eine Softwarekomponente."""
    suchtext = component.strip().lower()
    treffer = [host for host in HOSTS if suchtext in host["component"].lower()]
    if not suchtext or not treffer:
        bekannt = ", ".join(sorted({host["component"] for host in HOSTS}))
        return f"ERROR: no host runs '{component}'. Known components: {bekannt}"
    return " ; ".join(
        f"{host['host']} runs {host['component']} {host['version']} "
        f"({host['environment']}, {host['exposure']})"
        for host in treffer
    )


In [ ]:
# ▶️ Werkzeuge unabhängig vom Modell testen
zeige(cve_lookup("CVE-2026-3224"), titel="cve_lookup")
zeige(host_inventory("Devolutions Server"), titel="host_inventory")


---
## 3 · Challenge 1 — Der ReAct-Vertrag

Der System Prompt ist die Schnittstelle zwischen Modell und Programm. Formuliere `SYSTEM_REACT` auf Englisch. Er muss enthalten:

- beide Werkzeuge mit Signatur und Beispielargument,
- exakt die Schlüsselwörter `Thought:`, `Action:`, `Observation:` und `Final Answer:`,
- die Regel, dass nur das Programm Observations schreibt,
- den Ablauf: erst CVE nachschlagen, dann die gefundene Komponente im Inventar suchen,
- höchstens einen Aufruf je Werkzeug und eine ehrliche Final Answer bei `ERROR`.

Halte ihn kompakt. Ein vollständiger Beispieldurchlauf ist nicht nötig und könnte seine Beispielwerte später in die echte Antwort hineintragen.


In [ ]:
# 🛠️ Challenge 1: Formuliere den ReAct-Vertrag.
SYSTEM_REACT = """
"""

if not SYSTEM_REACT.strip():
    raise NotImplementedError("Formuliere den System Prompt.")


In [ ]:
# ✅ Selbsttest Challenge 1
assert len(SYSTEM_REACT.split()) >= 70
for begriff in ["Thought:", "Action:", "Observation", "Final Answer:",
                "cve_lookup", "host_inventory", "ERROR"]:
    assert begriff in SYSTEM_REACT, f"Im System Prompt fehlt {begriff!r}"
assert "Never" in SYSTEM_REACT or "never" in SYSTEM_REACT
print("✅ Challenge 1 gelöst")


---
## 4 · Challenge 2 — Vom Modelltext zum sicheren Werkzeugaufruf

Das Modell liefert beispielsweise `Action: cve_lookup(CVE-2026-3224)`. Python muss daraus Werkzeugname und Argument lesen. Erst danach entscheidet eine feste Registry, ob dieses Werkzeug überhaupt ausgeführt werden darf.

Implementiere:

- `parse_action(text)` → `(name, argument)` oder `(None, None)`. Akzeptiere sowohl
  `tool(value)` als auch die häufige Modellvariante `tool(parameter="value")`,
- `fuehre_aus(name, argument)` → Werkzeugergebnis oder einen `ERROR`-Text.

Unbekannte Namen werden niemals mit `eval()` ausgeführt.


In [ ]:
# ▶️ Nur diese Funktionen darf das Modell aufrufen
WERKZEUGE = {
    "cve_lookup": cve_lookup,
    "host_inventory": host_inventory,
}

TEST_AUSGABEN = [
    ("Thought: First get the facts.\nAction: cve_lookup(CVE-2026-3224)",
     ("cve_lookup", "CVE-2026-3224")),
    ('Action: host_inventory("Devolutions Server")',
     ("host_inventory", "Devolutions Server")),
    ('Action: cve_lookup(cve_id="CVE-2026-3224")',
     ("cve_lookup", "CVE-2026-3224")),
    ("**Action:** cve_lookup(CVE-2026-3224)", ("cve_lookup", "CVE-2026-3224")),
    ("Final Answer: I cannot verify this.", (None, None)),
]


In [ ]:
# 🛠️ Challenge 2: Implementiere Parsing und sichere Ausführung.
def parse_action(text):
    """Liest Werkzeugname und Argument aus einer Action-Zeile."""
    # TODO: Gib (name, argument) oder (None, None) zurück.
    raise NotImplementedError


def fuehre_aus(name, argument):
    """Führt nur registrierte Werkzeuge aus; Fehler werden zur Observation."""
    # TODO: Schlage name in WERKZEUGE nach und gib bei unbekannten Namen ERROR zurück.
    raise NotImplementedError


In [ ]:
# ✅ Selbsttest Challenge 2
for ausgabe, erwartet in TEST_AUSGABEN:
    assert parse_action(ausgabe) == erwartet, (ausgabe, parse_action(ausgabe), erwartet)

assert "CVE-2026-3224" in fuehre_aus("cve_lookup", "CVE-2026-3224")
assert "srv-pam01" in fuehre_aus("host_inventory", "Devolutions Server")
assert fuehre_aus("delete_files", "/").startswith("ERROR")
print("✅ Challenge 2 gelöst")


---
## 5 · Der kontrollierte Modellschritt

Für jeden Schleifendurchlauf brauchen wir genau eine Action oder Final Answer. Zwei technische Leitplanken sorgen dafür:

- `reasoning_effort=REASONING` verhindert beim konfigurierten kleinen Reasoning-Modell leeren Antworttext.
- `stop=["Observation:"]` beendet die Generierung, bevor das Modell eine Werkzeugantwort erfinden kann.

Diese Details gehören in eine zentrale Funktion, nicht verteilt in die Schleife.


In [ ]:
# ▶️ Genau ein kontrollierter Modellschritt
REASONING_ZUSATZ = {"reasoning_effort": REASONING} if REASONING else {}


def frage_schritt(nachrichten, max_tokens=300):
    antwort = client.chat.completions.create(
        model=MODELL,
        messages=nachrichten,
        temperature=0.0,
        max_tokens=max_tokens,
        stop=["Observation:"],
        **REASONING_ZUSATZ,
    )
    return (antwort.choices[0].message.content or "").strip()


def notiere(protokoll, zeile):
    protokoll.append(zeile)
    for absatz in zeile.splitlines():
        print(textwrap.fill(absatz, width=88, subsequent_indent="        "))


---
## 6 · Challenge 3 — Der vollständige Agent-Loop

Implementiere `react(frage, max_schritte=5)`:

1. Starte mit System Prompt und Frage.
2. Hole einen Modellschritt und speichere ihn als Assistant-Nachricht.
3. Bei `Final Answer:` ist der Lauf fertig.
4. Sonst: Action parsen, registriertes Werkzeug ausführen und das Ergebnis als `Observation: ...` in einer User-Nachricht zurückgeben.
5. Weise eine vorzeitige Final Answer zurück, wenn der CVE-Lookup erfolgreich war, aber das Inventar noch nicht geprüft wurde.
6. Brich nach `max_schritte` oder beim zweiten identischen Aufruf in Folge ab. Jedes Werkzeug darf höchstens einmal wirklich ausgeführt werden.

Die Grenzen machen eine falsche Strategie nicht richtig. Sie verhindern aber unkontrollierte Schleifen und unnötige Aufrufe.


In [ ]:
# 🛠️ Challenge 3: Implementiere den ReAct-Loop.
def react(frage, max_schritte=5):
    """Führt den ReAct-Loop aus und gibt seine Protokollzeilen zurück."""
    # TODO: Baue Nachrichten, Modellaufruf, Tool-Ausführung, Observation und Grenzen zusammen.
    raise NotImplementedError


In [ ]:
# ✅ Selbsttest Challenge 3 — deterministisch, ohne Modell
def skript_modell(ausgaben):
    rest = list(ausgaben)
    def antworte(nachrichten, max_tokens=300):
        return rest.pop(0)
    return antworte


SAUBER = [
    "Thought: I need the CVE facts.\nAction: cve_lookup(CVE-2026-3224)",
    "Thought: I need our affected hosts.\nAction: host_inventory(Devolutions Server)",
    "Thought: I can now answer.\nFinal Answer: srv-pam01 and srv-pam02 require an upgrade.",
]
WIEDERHOLUNG = [
    "Thought: Look it up.\nAction: cve_lookup(CVE-2026-9999)",
    "Thought: Try again.\nAction: cve_lookup(CVE-2026-9999)",
]
VORZEITIG = [
    "Thought: Get the CVE.\nAction: cve_lookup(CVE-2026-3224)",
    "Thought: I can answer.\nFinal Answer: Upgrade the product.",
    "Thought: I still need our hosts.\nAction: host_inventory(Devolutions Server)",
    "Thought: Now I can answer.\nFinal Answer: srv-pam01 and srv-pam02 are affected.",
]

echter_schritt = frage_schritt
try:
    frage_schritt = skript_modell(SAUBER)
    sauber = react(FRAGE)
    print()
    frage_schritt = skript_modell(WIEDERHOLUNG)
    wiederholung = react(FRAGE)
    print()
    frage_schritt = skript_modell(VORZEITIG)
    vorzeitig = react(FRAGE)
finally:
    frage_schritt = echter_schritt

assert len(sauber) == 5 and "Final Answer:" in sauber[-1]
assert "critical" in sauber[1] and "srv-pam01" in sauber[3]
assert len(wiederholung) == 4 and "ABBRUCH" in wiederholung[-1]
assert len(vorzeitig) == 7 and "premature" in vorzeitig[3]
assert "srv-pam01" in vorzeitig[-1]
print()
print("✅ Challenge 3 gelöst")


---
## 7 · Der echte Lauf

Jetzt arbeitet dieselbe getestete Schleife mit dem echten Modell. Lies das Protokoll als Belegkette: Jede Tatsachenbehauptung der Final Answer sollte auf eine Observation zurückgehen.


In [ ]:
# ▶️ ReAct mit dem echten Modell
protokoll = react(FRAGE)

print()
print("Kurzprüfung:")
gesamt = "\n".join(protokoll)
print("nennt betroffene Hosts:", "srv-pam01" in gesamt and "srv-pam02" in gesamt)
print("nennt feste Version:   ", CVES["CVE-2026-3224"]["fixed_version"] in gesamt)
print("endet kontrolliert:    ", "Final Answer:" in gesamt or "ABBRUCH" in protokoll[-1])


---
## 8 · Fazit

Ein ReAct-Agent besteht aus wenigen, klar getrennten Teilen:

1. Der **System Prompt** definiert Werkzeugvertrag und Ausgabeformat.
2. Der **Parser** übersetzt Modelltext in einen strukturierten Aufruf.
3. Eine feste **Registry** begrenzt, was tatsächlich ausgeführt werden darf.
4. Das Programm erzeugt die **Observation** aus echten Werkzeugdaten.
5. Der **Loop** wiederholt den Ablauf bis zur Final Answer oder einer Sicherheitsgrenze.

Der wichtigste Unterschied zu einem normalen Prompt: Das Modell liefert nicht allein die Antwort. Es steuert einen Prozess, dessen Datenzugriff und Grenzen weiterhin im Programm liegen.

### Transferfrage

Welche zusätzliche Prüfung bräuchtest du, bevor eine Final Answer in deinem eigenen Kontext automatisch eine reale Aktion auslösen dürfte?
